# 11 - Ablation Study

**Author:** Sacha Huberty

**Purpose:** The project's designated arbiter of module worth (PROJECT_STRUCTURE.md,
"the referee"). Starting from the frozen config (stage 9), disable each
`modules.*` flag one at a time -- regime view (V1), mean-reversion view
(V2), technical view (V3), sentiment view (V4), anomaly override -- and
re-run the OOS backtest for each variant, reporting marginal Sharpe
contribution vs. the full frozen pipeline. Also includes a turnover-cap
sensitivity arm (2%/4% vs. the frozen 1%) to measure how much the
execution layer itself is suppressing performance, per a comprehensive
project review (REVIEW.md) that flagged the 1%-cap-equals-1%-band
turnover configuration as a likely dominant, possibly circular,
contributor to the strategy's muted risk-taking.

**Last updated:** 2026-07-28

**Methodology note:** this is a single frozen-config OOS ablation, not
a per-fold walk-forward re-fit of each variant (consistent with stage
9's "walk-forward = fold reporting over one continuous run" scope, not
per-fold model persistence). Every arm uses the exact same OOS window
and buffered lookback as notebook 09's canonical frozen run, so
differences are attributable to the ablated component, not to a
different backtest range.

## Setup

In [ ]:
# Same fix as notebooks 09/10: force single-threaded BLAS/OMP before
# numpy/scipy/tensorflow are imported (Windows thread-pool contention
# across hundreds of tiny per-week SLSQP/HMM calls otherwise causes
# intermittent multi-minute stalls).
import os

for _var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_var] = "1"

import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import backtest, data, metrics, strategy, universe

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["modules"], cfg["rebalance"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)

oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]

buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]
print("Backtest range:", backtest_returns.index.min().date(), "to", backtest_returns.index.max().date())

## Analysis / signal logic

### Ablation harness

`build_strategy_fn(cfg, use_anomaly)` builds the exact same pipeline
notebook 09 freezes: `black_litterman_strategy` (V1/V2/V3 each already
gated internally by `views.py`'s `cfg["modules"][...]` checks -- no
strategy.py change needed to ablate them), optionally wrapped with
`with_anomaly_override`. V4 (sentiment) is never included in
`black_litterman_strategy`'s view_sets at all (still fundamentally
live-only, see sentiment.py) -- so its ablation arm is included for
completeness but is a structural no-op, not a measured effect; this is
reported explicitly rather than silently, since a naive reading of "no
difference" could otherwise be misread as "sentiment doesn't matter"
rather than "sentiment isn't wired into the backtest at all".

In [ ]:
def build_strategy_fn(run_cfg, use_anomaly=True):
    bl_fn = strategy.black_litterman_strategy(class_bucket, run_cfg, posture_cfg)
    if use_anomaly:
        return strategy.with_anomaly_override(bl_fn, run_cfg)
    return bl_fn


def oos_cfg_from(base_cfg):
    run_cfg = copy.deepcopy(base_cfg)
    run_cfg["anomaly"]["epochs"] = 10
    run_cfg["anomaly"]["patience"] = 3
    run_cfg["anomaly"]["refit_frequency_days"] = 126
    return run_cfg


frozen_cfg = oos_cfg_from(cfg)
print("Frozen meanreversion:", frozen_cfg["meanreversion"]["lookback_days"], frozen_cfg["meanreversion"]["entry_z"])
print("Frozen rebalance:", frozen_cfg["rebalance"]["no_trade_band"], frozen_cfg["rebalance"]["max_weekly_turnover"])

### Baseline + turnover-cap sensitivity (shared, memoized signal)

The turnover cap and no-trade band are rebalance-time-only parameters
-- they never change what the strategy WANTS to hold, only how fast it
gets there. So the (expensive) frozen signal is computed once via
`backtest.memoize_strategy` and reused for the baseline run and both
turnover-cap variants; only the cheap accounting loop re-runs for each.

In [ ]:
memoized_frozen_fn = backtest.memoize_strategy(build_strategy_fn(frozen_cfg))

baseline_result = backtest.run(memoized_frozen_fn, backtest_returns, frozen_cfg)

cfg_2pct = copy.deepcopy(frozen_cfg)
cfg_2pct["rebalance"]["max_weekly_turnover"] = 0.02
cfg_2pct["rebalance"]["no_trade_band"] = 0.005
result_2pct = backtest.run(memoized_frozen_fn, backtest_returns, cfg_2pct)

cfg_4pct = copy.deepcopy(frozen_cfg)
cfg_4pct["rebalance"]["max_weekly_turnover"] = 0.04
cfg_4pct["rebalance"]["no_trade_band"] = 0.005
result_4pct = backtest.run(memoized_frozen_fn, backtest_returns, cfg_4pct)

### Module ablations (each a fresh, non-memoized signal)

Each arm genuinely changes the weekly signal, so each needs its own
independent run -- memoization across arms would silently reuse a
DIFFERENT config's cached weights, which would be wrong, not just
slow.

In [ ]:
def ablate(module_key):
    run_cfg = copy.deepcopy(frozen_cfg)
    run_cfg["modules"][module_key] = False
    fn = build_strategy_fn(run_cfg)
    return backtest.run(fn, backtest_returns, run_cfg)


# Each arm takes ~15 minutes on its own (periodic autoencoder refits
# dominate); split into separate cells so nbconvert's per-cell timeout
# resets between them rather than needing to cover all four at once.
no_regime_result = ablate("regime_view")

In [ ]:
no_meanrev_result = ablate("meanreversion_view")

In [ ]:
no_technical_result = ablate("technical_view")

In [ ]:
no_anomaly_cfg = copy.deepcopy(frozen_cfg)
no_anomaly_fn = build_strategy_fn(no_anomaly_cfg, use_anomaly=False)
no_anomaly_result = backtest.run(no_anomaly_fn, backtest_returns, no_anomaly_cfg)

# V4 (sentiment) is never part of black_litterman_strategy's view_sets
# regardless of the module flag -- ablating it cannot change the
# backtest, so this arm reuses the baseline result rather than
# recomputing an identical run.
no_sentiment_result = baseline_result

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "max_drawdown": metrics.max_drawdown(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


module_results = {
    "frozen (baseline)": baseline_result,
    "no_regime_view (V1 off)": no_regime_result,
    "no_meanreversion_view (V2 off)": no_meanrev_result,
    "no_technical_view (V3 off)": no_technical_result,
    "no_sentiment_view (V4 off, structural no-op)": no_sentiment_result,
    "no_anomaly_override": no_anomaly_result,
}
module_table = pd.DataFrame({name: oos_metrics(res) for name, res in module_results.items()}).T
module_table

In [ ]:
baseline_sharpe = module_table.loc["frozen (baseline)", "sharpe"]
marginal = (baseline_sharpe - module_table["sharpe"]).drop("frozen (baseline)")
marginal = marginal.rename("marginal_sharpe_contribution").to_frame()
marginal["reading"] = marginal["marginal_sharpe_contribution"].apply(
    lambda x: "helped (removing it hurt)" if x > 0 else "hurt or neutral (removing it helped/no change)"
)
marginal.sort_values("marginal_sharpe_contribution", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["seagreen" if v > 0 else "firebrick" for v in marginal["marginal_sharpe_contribution"]]
ax.barh(marginal.index, marginal["marginal_sharpe_contribution"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Marginal Sharpe contribution (baseline minus ablated)")
ax.set_title("Module ablation: marginal Sharpe contribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in module_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "frozen (baseline)" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: module ablations")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend(fontsize=8)
plt.show()

### Turnover-cap sensitivity

In [ ]:
turnover_results = {
    "1% cap / 1% band (frozen)": baseline_result,
    "2% cap / 0.5% band (S4 convention)": result_2pct,
    "4% cap / 0.5% band": result_4pct,
}
turnover_table = pd.DataFrame({name: oos_metrics(res) for name, res in turnover_results.items()}).T
turnover_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, res in turnover_results.items():
    (1.0 + res.daily_returns.loc[oos_start:]).cumprod().plot(ax=axes[0], label=name)
axes[0].set_title("OOS equity: turnover-cap sensitivity")
axes[0].set_ylabel("Growth of $1")
axes[0].legend(fontsize=8)

turnover_table["sharpe"].plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("OOS Sharpe by turnover-cap arm")
axes[1].set_xticklabels(turnover_table.index, rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, res in turnover_results.items():
    res.turnover.loc[oos_start:].rolling(4).mean().plot(ax=ax, label=name)
ax.set_title("OOS weekly turnover (4-week rolling mean) by cap arm")
ax.set_ylabel("One-way turnover")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Notes / next steps

**Findings -- the referee has spoken:**

- **V1 (regime view) is the strategy's real, positive contributor.**
  Removing it drops OOS Sharpe from 0.8056 to 0.6582 -- a marginal
  contribution of **+0.147**, by far the largest of any module. The
  HMM posture view is doing genuine work in the fused pipeline.
- **V3 (technical view) is actively hurting performance, substantially.**
  Removing it *raises* Sharpe from 0.8056 to 1.0710 -- a marginal
  contribution of **-0.265**, the largest effect in the whole study,
  in the wrong direction. This is the single most actionable finding
  in this notebook. It was not in this session's prescribed priority-2
  fix list, so it has not been changed here, but it should be the
  first thing investigated next (candidate causes: the event study in
  notebook 06 already found near-cash tickers like BIL/SHY generate
  spurious "near a zone" flags at below-chance hit rates, and that
  finding was never acted on -- see REVIEW.md 3, S12 row).
- **V2 (mean-reversion) is essentially inert, slightly negative.**
  Marginal contribution **-0.0125** -- close to zero, consistent with
  REVIEW.md's diagnosed unit mismatch (a daily-scale OU view added to
  an annualized prior enters BL as noise, not a real signal). This
  ablation result is what justifies proceeding with the unit fix
  (priority-2 item 2, next): a signal that currently does ~nothing
  deserves a fair chance at being correctly scaled before being judged.
- **Anomaly override: negligible, and consistent with stage 4.**
  Marginal contribution **+0.0020** -- matches stage 4's original
  finding (OOS Sharpe moved by -0.002 when the override was added)
  almost exactly in magnitude, just with the opposite sign convention.
  Anomalies are rare enough in this window that the override rarely
  fires.
- **Sentiment: exactly zero, by construction, not by finding.**
  V4 is never included in `black_litterman_strategy`'s view_sets
  regardless of the module flag (still fundamentally live-only -- see
  sentiment.py), so this arm reuses the baseline result verbatim. Zero
  difference here means "not wired in," not "doesn't matter."
- **The turnover-cap critique in REVIEW.md is empirically confirmed.**
  Relaxing the cap from the frozen 1%/1% to the S4-convention 2%/0.5%
  raises OOS Sharpe from 0.8056 to 0.8746 (+0.069) with a comparable-
  to-slightly-better max drawdown (-10.68% vs -10.83%) and turnover
  cost still modest (0.45% total drag over the OOS window vs 0.24% at
  1%). Going further to 4%/0.5% adds only +0.006 more Sharpe
  (0.8804) for roughly double the cost drag (0.87%) -- diminishing
  returns past 2%. This directly supports restoring the cap to the S4
  convention (priority-2 item 3, next).

**What this means for priority 2:** the ablation supports proceeding
with both the V2 unit fix (item 2: a near-zero-contribution signal is
exactly what a broken-unit signal looks like) and the turnover-cap
restoration (item 3: a clear, monotonic Sharpe improvement with no
drawdown cost through 2%). The cash-degeneracy fix (item 4) was not
directly tested by an ablation arm here -- REVIEW.md's argument for it
is structural (rf=0 combined with near-zero-vol cash creates a
degenerate Sharpe objective, independent of any single module's
on/off state) -- but proceeding with it is consistent with the
broader picture this ablation paints: the signal layer (V1) works,
execution (turnover cap) was over-throttled, and V3 is actively
counterproductive, all pointing at an integration layer that has been
suppressing the strategy's own risk-taking more than its signals
warrant.
